In [1]:
import numpy as np
import geopandas as gpd
import rasterio
import fiona
import os

from importlib_resources import files
from beak.experimental.utilities.io import save_raster
from beak.experimental.utilities.misc import replace_invalid_characters
from beak.experimental.utilities.raster_processing import _reproject_raster_core

BASE_PATH = files('beak.data')
BASE_RASTER_PATH = BASE_PATH / "BASE_RASTERS"
PATH_TEMPLATE_EXTENT = BASE_PATH / "RAW" / "geophysics" / "regional_scale" / "poco_southwest_nm" / "magnetic" / "Geophysics_Mag_RTP_SWNM.tif"
PATH_TEMPLATE_CRS = BASE_PATH / "RAW" / "templates" / "features_crs.gpkg"

TARGET_EPSG = 102008
TARGET_RES = 50

print(f"Raster template found: {os.path.exists(PATH_TEMPLATE_EXTENT)}")
print(f"CRS template found: {os.path.exists(PATH_TEMPLATE_CRS)}")

print("\nThe CRS template cointains following layers:")
for i, layer in enumerate(fiona.listlayers(PATH_TEMPLATE_CRS)):
    print(f"Layer {i+1}: {layer}")


Raster template found: True
CRS template found: True

The CRS template cointains following layers:
Layer 1: epsg_102008


Load template raster and CRS

In [2]:
template_raster = rasterio.open(PATH_TEMPLATE_EXTENT)
template_crs = gpd.read_file(PATH_TEMPLATE_CRS, layer="epsg_102008").crs

In [3]:
# Reproject and resample
base_raster, base_meta = _reproject_raster_core(raster=template_raster,
                                                target_crs=template_crs,
                                                target_resolution=TARGET_RES,
                                                resampling_method=None,
                                                resampling_mode="auto",
                                                snap_to_origin=(0,0))

# Set nodata elements
new_nodata_value = -99
base_raster = np.where(base_raster==template_raster.nodata, new_nodata_value, 1)


In [5]:
out_file = BASE_RASTER_PATH / f"EPSG_{TARGET_EPSG}_RES_{replace_invalid_characters(str(TARGET_RES))}_POCO_SOUTHWEST_NM.tif"
print(f"Saving raster to: {out_file}")


Saving raster to: S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\BASE_RASTERS\EPSG_102008_RES_50_POCO_SOUTHWEST_NM.tif


In [6]:
save_raster(out_file, base_raster, metadata=base_meta, dtype="int8", nodata_value=new_nodata_value)
